# Dataset Adequacy Analysis — Sargentii Cherry (CF Model)

Investigates whether the `GMU_Cherry_Japan_YS` dataset provides sufficient
coverage to fit a **chill-forcing (CF)** phenology model across two cherry
species (*P. yedoensis*, *P. sargentii*).

Key questions:
1. **Cross-factor support** — are species confounded with region/climate?
2. **Temporal variation** — do the observed years span meaningfully different climates?
3. **Driver range** — do chilling and forcing accumulation vary enough to constrain the model?

Set `USE_SYNTHETIC = True` to replace the real dataset with controlled synthetic data
for debugging or to demonstrate what adequate/inadequate coverage looks like.

## Config

In [ ]:
# ── toggle real vs synthetic ───────────────────────────────────────────────
USE_SYNTHETIC = False

# Synthetic-mode settings
SYN_N_SPECIES  = 2       # number of species
SYN_N_LOCS     = 20      # locations per species
SYN_N_YEARS    = 20      # years per location
SYN_CONFOUNDED = True    # True → species A only cold sites, B only warm sites
                          # False → both species at all sites (good coverage)

# ── dataset & model settings ───────────────────────────────────────────────
DATASET_KEY = 'GMU_Cherry_Japan_YS'
OBS_KEY     = 'gmu_0'
CUTOFF      = 2006       # train / test split year

# ── climate binning ────────────────────────────────────────────────────────
# mean winter temperature (Oct–Jan) used for climate bins
CLIMATE_BINS   = [-10, -5, 0, 5, 10, 15, 20, 30]
CLIMATE_LABELS = ['<-5', '-5–0', '0–5', '5–10', '10–15', '15–20', '>20']

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from typing import Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from pysephone.constants import KEY_FEATURES, KEY_OBSERVATIONS_INDEX
from pysephone.dataset.dataset import Dataset
from pysephone.dataset.util.calendar import Calendar
from pysephone.dataset.util.openmeteo import OpenMeteoFeatures
from pysephone.data.gmu_cherry.regions_data import (
    LOCATIONS_REGIONS_JAPAN,
    REGIONS_JAPAN,
    LOCATIONS_JAPAN_SARGENTII,
    LOCATIONS_JAPAN_YEDOENSIS,
)
from pysephone.models.util.func_phenology import func_chilling_days, func_dynamic_chill_daily

SPECIES_NAMES = {0: 'yedoensis', 1: 'sargentii'}
SPECIES_COLORS = {0: '#2980b9', 1: '#e74c3c'}

def _region_name(loc_id: str) -> str:
    """Map a loc_id (double-underscore form) back to a REGIONS_JAPAN name."""
    loc_key = loc_id.replace('__', '/')
    region_id = LOCATIONS_REGIONS_JAPAN.get(loc_key)
    if region_id is None:
        return 'Unknown'
    return REGIONS_JAPAN.get(region_id, 'Unknown')

## 1. Load data

In [ ]:
if not USE_SYNTHETIC:
    cal      = Calendar(default_start='10-01', default_length=365)
    features = OpenMeteoFeatures(calendar=cal)
    ds = Dataset.load(DATASET_KEY, calendar=cal, feature_providers=[features])
    ds.download_features(verbose=True)
    print(f'Loaded {DATASET_KEY}: {len(list(ds.iter_items()))} samples')

else:
    # ── synthetic dataset ────────────────────────────────────────────────
    # Build a list of sample dicts directly; no Dataset wrapper needed.
    rng = np.random.default_rng(0)

    # Cold sites (mean winter T ~ -2°C) and warm sites (mean winter T ~ 8°C)
    cold_lats = rng.uniform(43, 46, SYN_N_LOCS)   # northern
    warm_lats = rng.uniform(34, 38, SYN_N_LOCS)   # southern
    cold_lons = rng.uniform(140, 145, SYN_N_LOCS)
    warm_lons = rng.uniform(130, 137, SYN_N_LOCS)

    def _syn_temp(mean_winter, mean_spring, n=365, seed=0):
        r = np.random.default_rng(seed)
        ts = np.zeros(n)
        ts[:92]    = mean_winter              # Oct–Dec
        ts[92:181] = mean_winter + 2          # Jan–Mar
        ts[181:273] = mean_spring             # Apr–Jun
        ts[273:]    = mean_spring + 5         # Jul–Sep
        return (ts + r.normal(0, 2, n)).astype(np.float32)

    def _syn_sample(species_id, lat, lon, year, mean_winter, mean_spring):
        ts = _syn_temp(mean_winter, mean_spring, seed=hash((species_id, year, round(lat, 1))) % 2**31)
        # Simple bloom: first day spring forcing exceeds threshold after chilling
        bloom_ix = int(180 - mean_winter * 2 + rng.normal(0, 3))
        bloom_ix = max(100, min(280, bloom_ix))
        return {
            'src': 'synthetic',
            'loc_id': f'loc_{species_id}_{round(lat,1)}',
            'year': year,
            'species_id': species_id,
            'lat': lat,
            'lon': lon,
            'region': 'Hokkaido' if mean_winter < 2 else ('Tohoku' if mean_winter < 6 else 'Kanto-Koshin'),
            'bloom_ix': bloom_ix,
            'mean_winter_T': float(ts[:92].mean()),
            'mean_spring_T': float(ts[181:273].mean()),
            'chill_days': float(func_chilling_days(ts).sum()),
            KEY_FEATURES: {
                'temperature_2m_mean': ts,
                'daylight_duration': (12.0 * 3600 * np.ones(365)).astype(np.float32),
            },
        }

    synthetic_samples = []
    years = list(range(2000, 2000 + SYN_N_YEARS))
    for year in years:
        for i in range(SYN_N_LOCS):
            if SYN_CONFOUNDED:
                # Species A only at cold sites, species B only at warm sites
                synthetic_samples.append(_syn_sample(0, warm_lats[i], warm_lons[i], year,
                                                     mean_winter=rng.normal(8, 1.5),
                                                     mean_spring=rng.normal(15, 1)))
                synthetic_samples.append(_syn_sample(1, cold_lats[i], cold_lons[i], year,
                                                     mean_winter=rng.normal(-2, 1.5),
                                                     mean_spring=rng.normal(8, 1)))
            else:
                # Both species at all sites
                mw = rng.normal(3, 4)
                ms = rng.normal(12, 2)
                lat = rng.uniform(34, 46)
                lon = rng.uniform(130, 145)
                synthetic_samples.append(_syn_sample(0, lat, lon, year, mw, ms))
                synthetic_samples.append(_syn_sample(1, lat, lon, year, mw, ms))

    print(f'Generated {len(synthetic_samples)} synthetic samples '
          f'({"confounded" if SYN_CONFOUNDED else "balanced"})')

In [ ]:
# ── Build summary dataframe ───────────────────────────────────────────────
if not USE_SYNTHETIC:
    rows = []
    for s in ds.iter_items():
        if OBS_KEY not in s.get('observations_index', {}):
            continue
        ts  = s[KEY_FEATURES]['temperature_2m_mean'].astype(float)
        rows.append({
            'loc_id':       s['loc_id'],
            'year':         s['year'],
            'species_id':   s['species_id'],
            'species':      SPECIES_NAMES.get(s['species_id'], str(s['species_id'])),
            'lat':          s.get('lat', np.nan),
            'lon':          s.get('lon', np.nan),
            'region':       _region_name(str(s['loc_id'])),
            'bloom_ix':     int(s['observations_index'][OBS_KEY]),
            'mean_winter_T': float(ts[:120].mean()),   # Oct–Jan
            'mean_spring_T': float(ts[120:212].mean()),  # Feb–May
            'chill_days':   float(func_chilling_days(ts).sum()),
            'forcing_gdu':  float(np.maximum(ts - 4.0, 0).sum()),
        })
    df = pd.DataFrame(rows)
else:
    df = pd.DataFrame([
        {
            'loc_id':       s['loc_id'],
            'year':         s['year'],
            'species_id':   s['species_id'],
            'species':      SPECIES_NAMES.get(s['species_id'], str(s['species_id'])),
            'lat':          s['lat'],
            'lon':          s['lon'],
            'region':       s['region'],
            'bloom_ix':     s['bloom_ix'],
            'mean_winter_T': s['mean_winter_T'],
            'mean_spring_T': s['mean_spring_T'],
            'chill_days':   s['chill_days'],
            'forcing_gdu':  float(np.maximum(
                s[KEY_FEATURES]['temperature_2m_mean'].astype(float) - 4.0, 0).sum()),
        }
        for s in synthetic_samples
    ])

df['climate_bin'] = pd.cut(df['mean_winter_T'], bins=CLIMATE_BINS, labels=CLIMATE_LABELS)

print(f'{len(df)} samples | {df["loc_id"].nunique()} locations | '
      f'{df["year"].nunique()} years | {df["species_id"].nunique()} species')
print()
print(df.groupby('species')['bloom_ix'].describe().round(1))

## 2. Spatial coverage

Map each observation coloured by species. Spatial clustering of species
is the first visual sign of confounding.

In [ ]:
loc_df = df.groupby(['loc_id', 'species', 'species_id']).agg(
    lat=('lat', 'first'), lon=('lon', 'first'),
    n=('bloom_ix', 'count'), bloom_mean=('bloom_ix', 'mean')
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Station coverage by species', fontsize=12, fontweight='bold')

# — map ——————————————————————————————————————————————————————————————————
ax = axes[0]
for sp_id, sp_name in SPECIES_NAMES.items():
    sub = loc_df[loc_df['species_id'] == sp_id]
    if sub.empty:
        continue
    ax.scatter(sub['lon'], sub['lat'], c=SPECIES_COLORS[sp_id],
               s=sub['n'] * 2, alpha=0.8, label=f'P. {sp_name} (n={len(sub)})',
               edgecolors='white', linewidths=0.4)
ax.set_xlabel('Longitude', fontsize=9)
ax.set_ylabel('Latitude', fontsize=9)
ax.set_title('Station map  (marker size ∝ number of years)', fontsize=9, fontweight='bold')
ax.legend(fontsize=8)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# — latitude distribution ——————————————————————————————————————————————
ax = axes[1]
bins = np.linspace(df['lat'].min() - 1, df['lat'].max() + 1, 25)
for sp_id, sp_name in SPECIES_NAMES.items():
    sub = df[df['species_id'] == sp_id]['lat']
    if sub.empty:
        continue
    ax.hist(sub, bins=bins, alpha=0.6, color=SPECIES_COLORS[sp_id],
            label=f'P. {sp_name}', edgecolor='white')
ax.set_xlabel('Latitude', fontsize=9)
ax.set_ylabel('Sample count', fontsize=9)
ax.set_title('Latitude distribution by species', fontsize=9, fontweight='bold')
ax.legend(fontsize=8)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

## 3. Cross-factor coverage

Coverage matrices showing sample counts for each (species, factor) pair.
A good dataset should have non-zero counts in most cells.
A nearly **rank-1** matrix signals that species and environment are confounded.

In [ ]:
def _heatmap(ax, mat: pd.DataFrame, title: str, fmt: str = 'd',
              cmap: str = 'YlOrRd', rank_label: bool = True):
    """Plot a count matrix as a heatmap with SVD rank annotation."""
    vals = mat.fillna(0).values.astype(float)
    im = ax.imshow(vals, cmap=cmap, aspect='auto')
    plt.colorbar(im, ax=ax, shrink=0.8, label='Sample count')
    ax.set_xticks(range(mat.shape[1]))
    ax.set_yticks(range(mat.shape[0]))
    ax.set_xticklabels(mat.columns, rotation=45, ha='right', fontsize=7.5)
    ax.set_yticklabels(mat.index, fontsize=8)
    for i in range(vals.shape[0]):
        for j in range(vals.shape[1]):
            v = int(vals[i, j])
            ax.text(j, i, str(v) if v > 0 else '',
                    ha='center', va='center', fontsize=7, color='black')
    if rank_label and vals.max() > 0:
        _, s, _ = np.linalg.svd(vals / (vals.sum() + 1e-9))
        explained = s[0]**2 / (s**2).sum()
        ax.set_title(f'{title}\n(top SV explains {explained:.0%} of variance)',
                     fontsize=9, fontweight='bold')
    else:
        ax.set_title(title, fontsize=9, fontweight='bold')


# ── species × region ——————————————————————————————————————————————————
mat_region = df.pivot_table(index='species', columns='region',
                             values='bloom_ix', aggfunc='count', fill_value=0)

# ── species × climate bin ——————————————————————————————————————————————
mat_climate = df.pivot_table(index='species', columns='climate_bin',
                              values='bloom_ix', aggfunc='count', fill_value=0)

# ── species × year (sampled; show last 20 years) ———————————————————————
recent_years = sorted(df['year'].unique())[-20:]
mat_year = (df[df['year'].isin(recent_years)]
            .pivot_table(index='species', columns='year',
                         values='bloom_ix', aggfunc='count', fill_value=0))

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
fig.suptitle('Cross-factor sample coverage  (rank-1 → confounded)',
             fontsize=12, fontweight='bold')

_heatmap(axes[0], mat_region,  'species × region')
axes[0].set_xlabel('Region', fontsize=9)
axes[0].set_ylabel('Species', fontsize=9)

_heatmap(axes[1], mat_climate, 'species × climate bin (mean winter T)')
axes[1].set_xlabel('Winter temperature bin (°C)', fontsize=9)
axes[1].set_ylabel('Species', fontsize=9)

_heatmap(axes[2], mat_year,    'species × year  (last 20 years)', rank_label=False)
axes[2].set_xlabel('Year', fontsize=9)
axes[2].set_ylabel('Species', fontsize=9)

plt.tight_layout()
plt.show()

# ── numeric rank diagnostic ————————————————————————————————————————————
print('Singular value spectra (normalised):')
for name, mat in [('species × region', mat_region),
                   ('species × climate', mat_climate)]:
    vals = mat.fillna(0).values.astype(float)
    if vals.max() == 0:
        continue
    _, s, _ = np.linalg.svd(vals / vals.sum())
    total = (s**2).sum()
    print(f'  {name}:')
    for i, sv in enumerate(s[:min(4, len(s))]):
        print(f'    SV{i+1}: {sv**2/total:.1%}')

## 4. Climate distribution by species

Overlapping distributions of the two key CF-model drivers — winter chilling
and spring forcing — by species. Separation here means species and climate
are confounded: the model cannot tell apart a species effect from a site effect.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('CF-model driver distributions by species',
             fontsize=12, fontweight='bold')

plot_vars = [
    ('mean_winter_T',  'Mean winter temperature (°C)',  'Oct–Jan mean T'),
    ('chill_days',     'Total chill days (T ≤ 7.2 °C)', 'Season chill days'),
    ('forcing_gdu',    'Total forcing GDU (T > 4 °C)',  'Season forcing GDU'),
]
for ax, (col, xlabel, title) in zip(axes, plot_vars):
    all_vals = df[col].dropna()
    bins = np.linspace(all_vals.min(), all_vals.max(), 28)
    for sp_id, sp_name in SPECIES_NAMES.items():
        sub = df[df['species_id'] == sp_id][col].dropna()
        if sub.empty:
            continue
        ax.hist(sub, bins=bins, alpha=0.55, color=SPECIES_COLORS[sp_id],
                label=f'P. {sp_name}', edgecolor='white', density=True)
        ax.axvline(sub.mean(), color=SPECIES_COLORS[sp_id], lw=1.5, ls='--')
    ax.set_xlabel(xlabel, fontsize=9)
    ax.set_ylabel('Density', fontsize=9)
    ax.set_title(title, fontsize=9, fontweight='bold')
    ax.legend(fontsize=8)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

# ── overlap coefficient ————————————————————————————————————————————————
print('Driver overlap between species (Bhattacharyya coefficient; 1.0 = identical distributions):')
sp_groups = [df[df['species_id'] == i] for i in sorted(SPECIES_NAMES.keys())]
if len(sp_groups) == 2:
    for col, label in [('mean_winter_T', 'Winter T'),
                        ('chill_days', 'Chill days'),
                        ('forcing_gdu', 'Forcing GDU')]:
        a, b = sp_groups[0][col].dropna(), sp_groups[1][col].dropna()
        bins = np.linspace(min(a.min(), b.min()), max(a.max(), b.max()), 30)
        ha, _ = np.histogram(a, bins=bins, density=True)
        hb, _ = np.histogram(b, bins=bins, density=True)
        bc = np.sum(np.sqrt(ha * hb)) * (bins[1] - bins[0])
        print(f'  {label}: {bc:.3f}')

## 5. Temporal variation

Multiple years are not useful if they are meteorologically similar.
We show:
- Bloom date trends over years per species
- Interannual temperature anomaly distributions (are there warm and cool years?)
- Identification of warm/cool years

In [ ]:
# ── per-year climatology ——————————————————————————————————————————————
yr_clim = df.groupby('year').agg(
    bloom_mean=('bloom_ix', 'mean'),
    bloom_std=('bloom_ix', 'std'),
    winter_T_mean=('mean_winter_T', 'mean'),
    spring_T_mean=('mean_spring_T', 'mean'),
    n=('bloom_ix', 'count'),
).reset_index()

yr_clim['winter_T_anom'] = yr_clim['winter_T_mean'] - yr_clim['winter_T_mean'].mean()
yr_clim['spring_T_anom'] = yr_clim['spring_T_mean'] - yr_clim['spring_T_mean'].mean()
yr_clim['bloom_anom']    = yr_clim['bloom_mean']    - yr_clim['bloom_mean'].mean()

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
fig.suptitle('Temporal variation — interannual anomalies', fontsize=12, fontweight='bold')

# — bloom date over time, by species ———————————————————————————————————
ax = axes[0, 0]
for sp_id, sp_name in SPECIES_NAMES.items():
    sub = df[df['species_id'] == sp_id].groupby('year')['bloom_ix'].median().reset_index()
    if sub.empty:
        continue
    ax.plot(sub['year'], sub['bloom_ix'], color=SPECIES_COLORS[sp_id],
            lw=1.8, marker='o', ms=3, label=f'P. {sp_name}')
ax.set_xlabel('Year', fontsize=9)
ax.set_ylabel('Median bloom day (from Oct 1)', fontsize=9)
ax.set_title('Bloom date over years', fontsize=9, fontweight='bold')
ax.legend(fontsize=8)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# — winter temperature anomaly ——————————————————————————————————————————
ax = axes[0, 1]
colors_bar = [('#e74c3c' if v > 0 else '#2980b9') for v in yr_clim['winter_T_anom']]
ax.bar(yr_clim['year'], yr_clim['winter_T_anom'], color=colors_bar, alpha=0.8)
ax.axhline(0, color='grey', lw=0.8)
ax.set_xlabel('Year', fontsize=9)
ax.set_ylabel('Winter T anomaly (°C)', fontsize=9)
ax.set_title('Winter temperature anomalies', fontsize=9, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# — spring temperature anomaly ——————————————————————————————————————————
ax = axes[1, 0]
colors_bar = [('#e74c3c' if v > 0 else '#2980b9') for v in yr_clim['spring_T_anom']]
ax.bar(yr_clim['year'], yr_clim['spring_T_anom'], color=colors_bar, alpha=0.8)
ax.axhline(0, color='grey', lw=0.8)
ax.set_xlabel('Year', fontsize=9)
ax.set_ylabel('Spring T anomaly (°C)', fontsize=9)
ax.set_title('Spring temperature anomalies', fontsize=9, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# — bloom anomaly vs winter T anomaly ——————————————————————————————————
ax = axes[1, 1]
sc = ax.scatter(yr_clim['winter_T_anom'], yr_clim['bloom_anom'],
                c=yr_clim['year'], cmap='viridis', s=50, zorder=5)
for _, row in yr_clim.iterrows():
    ax.annotate(str(int(row['year'])), (row['winter_T_anom'], row['bloom_anom']),
                fontsize=6.5, alpha=0.7, xytext=(2, 2), textcoords='offset points')
ax.axhline(0, color='grey', lw=0.7, ls='--')
ax.axvline(0, color='grey', lw=0.7, ls='--')
plt.colorbar(sc, ax=ax, label='Year')
r = yr_clim[['winter_T_anom', 'bloom_anom']].corr().iloc[0, 1]
ax.set_title(f'Bloom anomaly vs winter T anomaly  (r = {r:.2f})', fontsize=9, fontweight='bold')
ax.set_xlabel('Winter T anomaly (°C)', fontsize=9)
ax.set_ylabel('Bloom anomaly (days)', fontsize=9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

# ── warm / cool year summary ——————————————————————————————————————————
q33, q67 = yr_clim['winter_T_anom'].quantile([0.33, 0.67])
cool_years = yr_clim[yr_clim['winter_T_anom'] < q33]['year'].tolist()
warm_years = yr_clim[yr_clim['winter_T_anom'] > q67]['year'].tolist()
print(f'Cool winters (bottom third): {sorted(cool_years)}')
print(f'Warm winters (top third):    {sorted(warm_years)}')
print(f'Interannual winter T range:  {yr_clim["winter_T_mean"].min():.1f} – '
      f'{yr_clim["winter_T_mean"].max():.1f} °C '
      f'(spread {yr_clim["winter_T_mean"].max() - yr_clim["winter_T_mean"].min():.1f} °C)')

## 6. Driver range — can the model's parameters be constrained?

For a CF model to be identifiable, the data must include years where
chilling is **both** marginal (near-threshold) and ample. If all observations
have the same chilling surplus the threshold cannot be identified.

Similarly, forcing must vary so that `threshold_f` is constrained.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
fig.suptitle('Driver range analysis — identifiability of CF model parameters',
             fontsize=12, fontweight='bold')

# — chilling vs forcing scatter ————————————————————————————————————————
ax = axes[0]
for sp_id, sp_name in SPECIES_NAMES.items():
    sub = df[df['species_id'] == sp_id]
    if sub.empty:
        continue
    ax.scatter(sub['chill_days'], sub['forcing_gdu'],
               c=SPECIES_COLORS[sp_id], s=12, alpha=0.4, label=f'P. {sp_name}')
ax.set_xlabel('Total chill days (T ≤ 7.2 °C)', fontsize=9)
ax.set_ylabel('Total forcing GDU (T > 4 °C)', fontsize=9)
ax.set_title('Chilling vs forcing driver space', fontsize=9, fontweight='bold')
ax.legend(fontsize=8)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# — chill days vs bloom date ———————————————————————————————————————————
ax = axes[1]
for sp_id, sp_name in SPECIES_NAMES.items():
    sub = df[df['species_id'] == sp_id]
    if sub.empty:
        continue
    ax.scatter(sub['chill_days'], sub['bloom_ix'],
               c=SPECIES_COLORS[sp_id], s=12, alpha=0.4, label=f'P. {sp_name}')
    r = sub[['chill_days', 'bloom_ix']].corr().iloc[0, 1]
    ax.annotate(f'r={r:.2f}', xy=(0.05, 0.92 - sp_id * 0.08),
                xycoords='axes fraction', fontsize=8, color=SPECIES_COLORS[sp_id])
ax.set_xlabel('Total chill days', fontsize=9)
ax.set_ylabel('Bloom day (from Oct 1)', fontsize=9)
ax.set_title('Chill days vs bloom date', fontsize=9, fontweight='bold')
ax.legend(fontsize=8)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# — forcing GDU vs bloom date ——————————————————————————————————————————
ax = axes[2]
for sp_id, sp_name in SPECIES_NAMES.items():
    sub = df[df['species_id'] == sp_id]
    if sub.empty:
        continue
    ax.scatter(sub['forcing_gdu'], sub['bloom_ix'],
               c=SPECIES_COLORS[sp_id], s=12, alpha=0.4, label=f'P. {sp_name}')
    r = sub[['forcing_gdu', 'bloom_ix']].corr().iloc[0, 1]
    ax.annotate(f'r={r:.2f}', xy=(0.05, 0.92 - sp_id * 0.08),
                xycoords='axes fraction', fontsize=8, color=SPECIES_COLORS[sp_id])
ax.set_xlabel('Total forcing GDU', fontsize=9)
ax.set_ylabel('Bloom day (from Oct 1)', fontsize=9)
ax.set_title('Forcing GDU vs bloom date', fontsize=9, fontweight='bold')
ax.legend(fontsize=8)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

# ── driver range summary ——————————————————————————————————————————————
print('Driver ranges by species:')
for col, label in [('chill_days', 'Chill days'), ('forcing_gdu', 'Forcing GDU')]:
    print(f'  {label}:')
    for sp_id, sp_name in SPECIES_NAMES.items():
        sub = df[df['species_id'] == sp_id][col].dropna()
        if sub.empty:
            continue
        print(f'    P. {sp_name}: {sub.min():.0f} – {sub.max():.0f}  '
              f'(mean {sub.mean():.0f}, std {sub.std():.0f})')

## 7. Summary: adequacy verdict

Collate the key diagnostics into a single table.

In [ ]:
def _sv1_explained(mat: pd.DataFrame) -> float:
    vals = mat.fillna(0).values.astype(float)
    if vals.max() == 0:
        return float('nan')
    _, s, _ = np.linalg.svd(vals / vals.sum())
    return float(s[0]**2 / (s**2).sum())

n_years = df['year'].nunique()
n_locs  = df['loc_id'].nunique()
yr_range = yr_clim['winter_T_mean'].max() - yr_clim['winter_T_mean'].min()

sp_counts = df.groupby('species_id')['loc_id'].nunique()
both_species_n_locs = ', '.join(
    f'P. {SPECIES_NAMES.get(i, i)}: {n}' for i, n in sp_counts.items()
)

sv1_region  = _sv1_explained(mat_region)
sv1_climate = _sv1_explained(mat_climate)

rows = [
    ('Total samples',            len(df),             'Should be >> n_params'),
    ('Locations',                n_locs,              '—'),
    ('Years',                    n_years,             'Need interannual variation'),
    ('Stations per species',     both_species_n_locs, 'Both species need broad coverage'),
    ('Species × region  SV1',    f'{sv1_region:.0%}', '< 80% = adequate; ≥ 90% = confounded'),
    ('Species × climate  SV1',   f'{sv1_climate:.0%}','< 80% = adequate; ≥ 90% = confounded'),
    ('Interannual winter T range',f'{yr_range:.1f} °C','> 5 °C recommended for CF models'),
    ('Bloom date std (all)',      f'{df["bloom_ix"].std():.1f} days', 'Larger = more signal'),
]
display(pd.DataFrame(rows, columns=['Diagnostic', 'Value', 'Guideline']))